In [24]:
"""CriticalSpam - Preprocessing Module

This notebook preprocesses geospatial inputs (line trace, extents) and prepares
DEM download parameters. It is designed to be executed interactively or via
Papermill.

The notebook is Sphinx-compatible (NumPy docstring style).
"""

'CriticalSpam - Preprocessing Module\n\nThis notebook preprocesses geospatial inputs (line trace, extents) and prepares\nDEM download parameters. It is designed to be executed interactively or via\nPapermill.\n\nThe notebook is Sphinx-compatible (NumPy docstring style).\n'

### Notebook
- **Name:** `cs_preproc.ipynb`
- **Created/updated:** 2026-02-27
- **Python:** 3.x
- **Authors:** Mario Manana (mananam@unican.es)

### Purpose
Preprocess geospatial inputs (line trace, extents) and prepare DEM download parameters.

### Inputs
Line trace (`.shp`), `config.toml` parameters, WindNinja `fetch_dem` availability.

### Outputs
Derived extents and prepared inputs for DEM download.

### Dependencies
- (see imports below)

### Usage
Executed by the project pipeline (e.g., via Papermill) or run interactively in Jupyter.

### Notes
- Keep paths and parameters centralized in `config.toml` / `CONFIG_PATH` where applicable.


In [25]:
# Parameters (Papermill)
CONFIG_PATH = r"E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml" # "config.toml" #"config.toml"
PROJECT_ROOT = r"E:\mario\python\criticalspam"

### DEM (MDT) — preprocessing

This notebook uses the line trace file (`.shp`) to define:
- an axis-aligned bounding rectangle that encloses the line, and
- an expanded rectangle obtained by enlarging the first one by a factor `p` (defined in `config.toml`).


In [26]:
import sys
print("Python version")
print(sys.version)  
print("Executable")
print(sys.executable)

Python version
3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Executable
c:\Python\Python312\python.exe


In [27]:
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from dataclasses import dataclass
from pathlib import Path
from pyproj import Transformer
import subprocess
#import tomllib

In [28]:

# Add src folder to the path 
SRC = PROJECT_ROOT + r"\src"
print(f"Adding to sys.path: {SRC}")

# Print CONFIG_PATH 
print(f"Using config path: {CONFIG_PATH}")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# import custom utilities from src/cs_utils.py file
from cs_utils import Config, join_base, load_config_toml

Adding to sys.path: E:\mario\python\criticalspam\src
Using config path: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml


In [29]:
# Load configuration from the specified CONFIG_PATH
cfg = load_config_toml(CONFIG_PATH)

print("cs_preproc.ipynb")
print("Config path:", CONFIG_PATH)

cs_preproc.ipynb
Config path: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml


### Read parameters with `tomllib`


In [30]:

def iter_coords(geom):
    """Return the list of (x, y) points with all vertices of the geometry."""
    if geom is None or geom.is_empty:
        return []

    gt = geom.geom_type

    if gt == "Point":
        return [(geom.x, geom.y)]

    if gt in ("LineString", "LinearRing"):
        return list(geom.coords)

    if gt == "Polygon":
        coords = list(geom.exterior.coords)
        # Si quieres incluir agujeros interiores:
        # for ring in geom.interiors:
        #     coords.extend(list(ring.coords))
        return coords

    if gt.startswith("Multi") or gt == "GeometryCollection":
        coords = []
        for g in geom.geoms:
            coords.extend(iter_coords(g))
        return coords

    raise ValueError(f"Tipo geométrico no soportado: {gt}")

In [31]:
def utm_rect_to_fetch_dem_bbox(corners_xy, epsg_utm):
    """
    corners_xy: iterable con 4 tuplas (x, y) UTM [m]
    epsg_utm: EPSG del sistema UTM de entrada (p.ej. 25830, 32630)
    Devuelve: (north, east, south, west) en grados (lon/lat WGS84)
    """
    t = Transformer.from_crs(f"EPSG:{epsg_utm}", "EPSG:4326", always_xy=True)

    lons, lats = [], []
    for x, y in corners_xy:
        lon, lat = t.transform(x, y)
        lons.append(lon)
        lats.append(lat)

    north = max(lats)
    south = min(lats)
    east  = max(lons)
    west  = min(lons)

    print(f"Esquina NW: ({north:.6f}, {west:.6f})")
    print(f"Esquina SE: ({south:.6f}, {east:.6f})")
    print(f"Esquina NE: ({north:.6f}, {east:.6f})")
    print(f"Esquina SW: ({south:.6f}, {west:.6f})")

    return north, east, south, west

In [32]:
# --- Carga ---
gdf = gpd.read_file(cfg.in_shp)
gdf.crs

In [33]:
# Asegurar CRS EPSG:25830
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=25830)
elif gdf.crs.to_epsg() != 25830:
    gdf = gdf.to_crs(epsg=25830)
    
gdf.crs

<Projected CRS: EPSG:25830>
Name: ETRS89 / UTM zone 30N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Europe between 6°W and 0°W: Faroe Islands offshore; Ireland - offshore; Jan Mayen - offshore; Norway including Svalbard - offshore; Spain - mainland - onshore and offshore.
- bounds: (-6.0, 35.26, 0.01, 80.49)
Coordinate Operation:
- name: UTM zone 30N
- method: Transverse Mercator
Datum: European Terrestrial Reference System 1989 ensemble
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [34]:
# --- Extraer todos los vértices (con id de feature y nº de vértice) ---
rows = []
for fid, geom in zip(gdf.index, gdf.geometry):
    for vidx, (x, y, z) in enumerate(iter_coords(geom)):
        rows.append((fid, vidx, float(x), float(y), float(z)))


In [35]:
# --- Bounding box global ---
minx, miny, maxx, maxy = gdf.total_bounds

# --- Extraer todos los vértices (con id de feature y nº de vértice) ---
rows = []
for fid, geom in zip(gdf.index, gdf.geometry):
    for vidx, c in enumerate(iter_coords(geom)):
        x, y = c[0], c[1]          # toma solo las dos primeras componentes
        rows.append((fid, vidx, float(x), float(y)))

In [36]:
arr = np.array(rows, dtype=object)
xs = arr[:, 2].astype(float)
ys = arr[:, 3].astype(float)

tol = 1e-12

# Seleccionar los vértices que alcanzan extremos
pts = []
labels = []

def add_matches(mask, label):
    """
    add_matches.

    Notes
    -----
    Auto-generated docstring. Please refine parameter/return descriptions if needed.
    """
    for fid, vidx, x, y in arr[mask]:
        pts.append(Point(float(x), float(y)))
        labels.append((label, int(fid), int(vidx), float(x), float(y)))

add_matches(np.isclose(xs, minx, atol=tol), "x_min")
add_matches(np.isclose(xs, maxx, atol=tol), "x_max")
add_matches(np.isclose(ys, miny, atol=tol), "y_min")
add_matches(np.isclose(ys, maxy, atol=tol), "y_max")

# --- Construir GeoDataFrame de salida ---
out_gdf = gpd.GeoDataFrame(
    {
        "extremo": [t[0] for t in labels],
        "feat_id": [t[1] for t in labels],
        "vert_id": [t[2] for t in labels],
        "x":       [t[3] for t in labels],
        "y":       [t[4] for t in labels],
    },
    geometry=pts,
    crs="EPSG:25830"
)

# (Opcional) eliminar duplicados exactos (p.ej., si un punto cumple x_min y y_min)  
out_gdf = out_gdf.drop_duplicates(subset=["x", "y"]).reset_index(drop=True)

# --- Exportar a Shapefile ---
out_gdf.to_file(cfg.out_shp, driver="ESRI Shapefile", encoding="UTF-8")

print(f"Exportado: {cfg.out_shp}  (n={len(out_gdf)} puntos)")
print("BBox:", (minx, miny, maxx, maxy))


Exportado: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\Calculos\extremos_bbox.shp  (n=18 puntos)
BBox: (np.float64(257480.38), np.float64(4806678.5), np.float64(270788.98), np.float64(4808506.23))


In [37]:
# rows: lista de tuplas (fid, vidx, x, y)
arr = np.array(rows, dtype=object)
xs = arr[:, 2].astype(float)
ys = arr[:, 3].astype(float)

minx, maxx = xs.min(), xs.max()
miny, maxy = ys.min(), ys.max()

# Polígono del rectángulo (cerrado)
rect = Polygon([
    (minx, miny),
    (maxx, miny),
    (maxx, maxy),
    (minx, maxy),
    (minx, miny)
])

rect_gdf = gpd.GeoDataFrame(
    {"tipo": ["bbox_ejes"], "minx":[minx], "miny":[miny], "maxx":[maxx], "maxy":[maxy]},
    geometry=[rect],
    crs="EPSG:25830"
)

rect_gdf.to_file(cfg.out_rec_shp, driver="ESRI Shapefile", encoding="UTF-8")
print("Exportado: {cfg.out_rec_shp}")

Exportado: {cfg.out_rec_shp}


### Concentric expanded rectangle from an axis-aligned bounding box

- Start from the **axis-aligned** bounding rectangle defined by the global extrema of the trace:

  $$
  [x_{\min},x_{\max}] \times [y_{\min},y_{\max}]
  $$

  This is the classic (non-rotated) extent.

- Compute **width** and **height**:
  $$
  w=x_{\max}-x_{\min},\qquad h=y_{\max}-y_{\min}
  $$

- Compute the geometric **center** of the rectangle and expand about that center.


In [38]:
# --- Parámetro: incremento relativo (p.ej. 0.10 = +10%) ---
#p = 0.20

# --- Partimos de tus puntos (rows: (fid, vidx, x, y)) ---
arr = np.array(rows, dtype=object)
xs = arr[:, 2].astype(float)
ys = arr[:, 3].astype(float)

minx, maxx = xs.min(), xs.max()
miny, maxy = ys.min(), ys.max()

# Rectángulo original (bbox ejes)
rect0 = Polygon([
    (minx, miny), (maxx, miny), (maxx, maxy), (minx, maxy), (minx, miny)
])

# --- Rectángulo ampliado, centrado en el original ---
w = maxx - minx
h = maxy - miny
cx = 0.5 * (minx + maxx)
cy = 0.5 * (miny + maxy)

delta = ((w + h) / 2) * cfg.p

w2 = w + delta
h2 = h + delta

minx2 = cx - 0.5 * w2
maxx2 = cx + 0.5 * w2
miny2 = cy - 0.5 * h2
maxy2 = cy + 0.5 * h2

rect1 = Polygon([
    (minx2, miny2), (maxx2, miny2), (maxx2, maxy2), (minx2, maxy2), (minx2, miny2)
])

# --- Guardar ambos como una capa (dos polígonos) ---
out = gpd.GeoDataFrame(
    {
        "tipo": ["bbox_original", f"bbox_expand_{int(cfg.p*100)}pct"],
        "minx": [minx,  minx2],
        "miny": [miny,  miny2],
        "maxx": [maxx,  maxx2],
        "maxy": [maxy,  maxy2],
        "pct":  [0.0,   cfg.p],
    },
    geometry=[rect0, rect1],
    crs="EPSG:25830"
)

out.to_file(cfg.out_rec_exp_shp, driver="ESRI Shapefile", encoding="UTF-8")
print("Exportado: {cfg.out_rec_exp_shp}")


Exportado: {cfg.out_rec_exp_shp}


### Download the DEM covering the line corridor (WindNinja `fetch_dem`)

The DEM is downloaded using `fetch_dem.exe` provided by WindNinja.

**Windows check**
- Open `cmd` and run: `where fetch_dem`
- If no path is returned, add the WindNinja folder to the Windows `PATH` environment variable and restart the terminal/session.


In [39]:
# --- Ejemplo de uso ---
epsg_utm = 25830
corners_xy = [
    (minx2, miny2),
    (maxx2, miny2),
    (maxx2, maxy2),
    (minx2, maxy2),
]

north, east, south, west = utm_rect_to_fetch_dem_bbox(corners_xy, epsg_utm)

Esquina NW: (43.401451, -6.004005)
Esquina SE: (43.366746, -5.819809)
Esquina NE: (43.401451, -5.819809)
Esquina SW: (43.366746, -6.004005)


In [40]:
# --- Entradas ---
#epsg_utm = 25830          # cambia según tu caso (p.ej. 32630, 25829, etc.)
#x, y = 450000, 4800000    # tus coordenadas UTM (m)
#buf_x_km = 5              # semiancho (km)
#buf_y_km = 5              # semialto  (km)

# UTM -> lon/lat (WGS84 geográficas)
#t = Transformer.from_crs(f"EPSG:{epsg_utm}", "EPSG:4326", always_xy=True)
#lon, lat = t.transform(x, y)

# --- Llamada a fetch_dem ---
# Nota: el orden que exige fetch_dem es (lon, lat, buf_x, buf_y). :contentReference[oaicite:1]{index=1}
cmd = [
    "fetch_dem",
    #"--point", str(lon), str(lat), str(buf_x_km), str(buf_y_km),
    "--bbox", str(north), str(east), str(south), str(west),
    "--buf_units", "kilometers",
    "--src", "srtm",
    "--out_res", "30",
    "--fill_no_data",
    cfg.out_mdt_tif
]

p = subprocess.run(cmd, capture_output=True, text=True)

print("returncode:", p.returncode)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)

if p.returncode != 0:
    raise RuntimeError("fetch_dem failure; Please review STDERR.")

returncode: 0
STDOUT:
 
STDERR:
 
